In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import OrdinalEncoder,LabelEncoder, OneHotEncoder
from sklearn.preprocessing import MinMaxScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split 

import sklearn
print(sklearn.__version__)
import category_encoders
print(category_encoders.__version__)

1.9.1
2.11.1


## Преобразование колонок различных типов

In [2]:
import pandas as pd
X = pd.DataFrame(
    {'city': ['London', 'London', 'Paris', 'Sallisaw'],
     'title': ["His Last Bow", "How Watson Learned the Trick",
               "A Moveable Feast", "The Grapes of Wrath"],
     'expert_rating': [5, 3, 4, 5],
     'user_rating': [4, 5, 4, 3]})
X

,city,title,expert_rating,user_rating
0,London,His Last Bow,5,4
1,London,How Watson Learned the Trick,3,5
2,Paris,A Moveable Feast,4,4
3,Sallisaw,The Grapes of Wrath,5,3


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import make_column_selector


ct = ColumnTransformer([
      ('scale', StandardScaler(),
      make_column_selector(dtype_include=np.number)),
      ('onehot',
      OneHotEncoder(sparse_output=False),
      make_column_selector(pattern='city', dtype_include=object))
      ]).set_output(transform='pandas')

ct.fit_transform(X)

,scale__expert_rating,scale__user_rating,onehot__city_London,onehot__city_Paris,onehot__city_Sallisaw
0,0.904534,0.000000,1.0,0.0,0.0
1,-1.507557,1.414214,1.0,0.0,0.0
2,-0.301511,0.000000,0.0,1.0,0.0
3,0.904534,-1.414214,0.0,0.0,1.0


In [4]:
ct.get_feature_names_out()

array(['scale__expert_rating', 'scale__user_rating',
       'onehot__city_London', 'onehot__city_Paris',
       'onehot__city_Sallisaw'], dtype=object)

In [5]:
pd.DataFrame(data=ct.fit_transform(X), columns = ct.get_feature_names_out())

,scale__expert_rating,scale__user_rating,onehot__city_London,onehot__city_Paris,onehot__city_Sallisaw
0,0.904534,0.000000,1.0,0.0,0.0
1,-1.507557,1.414214,1.0,0.0,0.0
2,-0.301511,0.000000,0.0,1.0,0.0
3,0.904534,-1.414214,0.0,0.0,1.0


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import make_column_selector

display(X)

ct_r = ColumnTransformer([
      ('scale', StandardScaler(),
      make_column_selector(pattern='expert_rating')),
      ('onehot',
      OneHotEncoder(sparse_output=False),
      make_column_selector(pattern='city', dtype_include=object)),
      ('drop_colunm',
       'drop',
      make_column_selector(pattern='title'))
      ],
      remainder='passthrough').set_output(transform='pandas')

X_cl = ct_r.fit_transform(X)

X_cl

,city,title,expert_rating,user_rating
0,London,His Last Bow,5,4
1,London,How Watson Learned the Trick,3,5
2,Paris,A Moveable Feast,4,4
3,Sallisaw,The Grapes of Wrath,5,3


,scale__expert_rating,onehot__city_London,onehot__city_Paris,onehot__city_Sallisaw,remainder__user_rating
0,0.904534,1.0,0.0,0.0,4
1,-1.507557,1.0,0.0,0.0,5
2,-0.301511,0.0,1.0,0.0,4
3,0.904534,0.0,0.0,1.0,3


In [7]:
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import MinMaxScaler

display(X)

column_trans = make_column_transformer(
    (OneHotEncoder(sparse_output=False), ['city']),
    ('drop', 'title'),
    remainder=MinMaxScaler()).set_output(transform='pandas')

display(column_trans)

z = column_trans.fit_transform(X)
z

,city,title,expert_rating,user_rating
0,London,His Last Bow,5,4
1,London,How Watson Learned the Trick,3,5
2,Paris,A Moveable Feast,4,4
3,Sallisaw,The Grapes of Wrath,5,3


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('onehotencoder', ...), ('drop', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",MinMaxScaler()
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` a

,onehotencoder__city_London,onehotencoder__city_Paris,onehotencoder__city_Sallisaw,remainder__expert_rating,remainder__user_rating
0,1.0,0.0,0.0,1.0,0.5
1,1.0,0.0,0.0,0.0,1.0
2,0.0,1.0,0.0,0.5,0.5
3,0.0,0.0,1.0,1.0,0.0


In [8]:
X_col_tr = column_trans.fit_transform(X)
X_col_tr

,onehotencoder__city_London,onehotencoder__city_Paris,onehotencoder__city_Sallisaw,remainder__expert_rating,remainder__user_rating
0,1.0,0.0,0.0,1.0,0.5
1,1.0,0.0,0.0,0.0,1.0
2,0.0,1.0,0.0,0.5,0.5
3,0.0,0.0,1.0,1.0,0.0


## Сложные преобразования и заполнения пропущенных данных

In [9]:
df = pd.read_csv('Diamants/dim_train.csv')
df.tail()

,carat,cut,color,clarity,depth,table,price,x,y,z
37753,0.33,Premium,G,SI1,59.8,58.0,631,4.54,4.52,2.71
37754,2.01,Fair,F,SI1,58.6,95.0,13387,8.32,8.31,4.87
37755,0.26,Ideal,E,VVS2,61.5,56.0,769,4.11,4.08,2.52
37756,0.40,Very Good,G,SI1,62.8,60.0,702,4.66,4.70,2.94
37757,0.30,Ideal,D,SI1,62.4,54.0,508,4.32,4.34,2.70


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37758 entries, 0 to 37757
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   carat    37758 non-null  float64
 1   cut      37758 non-null  object 
 2   color    37758 non-null  object 
 3   clarity  37758 non-null  object 
 4   depth    37758 non-null  float64
 5   table    37758 non-null  float64
 6   price    37758 non-null  int64  
 7   x        37758 non-null  float64
 8   y        37758 non-null  float64
 9   z        37758 non-null  float64
dtypes: float64(6), int64(1), object(3)
memory usage: 2.9+ MB


In [11]:
df.describe()

,carat,depth,table,price,x,y,z
count,37758.000000,37758.000000,37758.000000,37758.000000,37758.000000,37758.000000,37758.000000
mean,0.798777,61.755085,57.460784,3932.113300,5.732871,5.736693,3.539717
std,0.474574,1.440063,2.230229,3980.261121,1.122451,1.154608,0.695749
min,0.200000,43.000000,43.000000,326.000000,0.000000,0.000000,0.000000
25%,0.400000,61.000000,56.000000,953.000000,4.720000,4.720000,2.910000
50%,0.700000,61.800000,57.000000,2404.000000,5.700000,5.710000,3.530000
75%,1.040000,62.500000,59.000000,5345.000000,6.540000,6.540000,4.040000
max,5.010000,79.000000,95.000000,18823.000000,10.740000,58.900000,8.060000


In [12]:
# отбор всех сторок с нулевым. значением в любом месте колонки
df[df.eq(0).any(axis=1)]

,carat,cut,color,clarity,depth,table,price,x,y,z
2949,1.07,Ideal,F,SI2,61.6,56.0,4954,0.00,6.62,0.0
6374,1.01,Premium,F,SI2,59.2,58.0,3837,6.50,6.47,0.0
8075,1.00,Very Good,H,VS2,63.3,53.0,5139,0.00,0.00,0.0
8909,2.80,Good,G,SI2,63.8,58.0,18788,8.90,8.85,0.0
9673,2.18,Premium,H,SI2,59.4,61.0,12631,8.49,8.45,0.0
12121,0.71,Good,F,SI2,64.1,60.0,2130,0.00,0.00,0.0
18790,1.10,Premium,G,SI2,63.0,59.0,3696,6.50,6.47,0.0
20612,0.71,Good,F,SI2,64.1,60.0,2130,0.00,0.00,0.0
21727,2.25,Premium,I,SI1,61.3,58.0,15397,8.52,8.42,0.0
25252,1.15,Ideal,G,VS2,59.2,56.0,5564,6.88,6.83,0.0


In [13]:
df.describe()

,carat,depth,table,price,x,y,z
count,37758.000000,37758.000000,37758.000000,37758.000000,37758.000000,37758.000000,37758.000000
mean,0.798777,61.755085,57.460784,3932.113300,5.732871,5.736693,3.539717
std,0.474574,1.440063,2.230229,3980.261121,1.122451,1.154608,0.695749
min,0.200000,43.000000,43.000000,326.000000,0.000000,0.000000,0.000000
25%,0.400000,61.000000,56.000000,953.000000,4.720000,4.720000,2.910000
50%,0.700000,61.800000,57.000000,2404.000000,5.700000,5.710000,3.530000
75%,1.040000,62.500000,59.000000,5345.000000,6.540000,6.540000,4.040000
max,5.010000,79.000000,95.000000,18823.000000,10.740000,58.900000,8.060000


In [14]:
df.head()

,carat,cut,color,clarity,depth,table,price,x,y,z
0,1.52,Ideal,J,SI1,61.9,57.0,7491,7.37,7.33,4.55
1,0.31,Very Good,F,SI2,63.0,58.0,408,4.34,4.36,2.74
2,1.00,Premium,F,SI1,60.7,60.0,5292,6.40,6.36,3.87
3,1.01,Very Good,E,VVS2,63.3,58.0,8912,6.39,6.31,4.02
4,0.40,Premium,I,SI2,62.9,59.0,585,4.68,4.63,2.93


Выделим отдельно категориальные фичи и те, к которым мы хотим сделать полиномиальный фичинг. Также отдельно остивим фичи, где будем применять StandartScaler. Также выделим целевую переменную

In [15]:
cat_features = ['cut','color']
axis_features = ['x','y','z']
num_features = ['carat','depth','table']
y = np.array(df.price)
X = df.drop(columns=['price'])

Разделим на train/test

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

## Расширенные преобразования в энкодоре

In [17]:
#### Нам нужна библиотека для строгой иерархической перекодировки https://contrib.scikit-learn.org/category_encoders/
#pip install category_encoders

In [18]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, make_column_transformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
import category_encoders as ce

Следующие три блока делают одно и тоже, просто признаки ставновятся в разный порядок, если их позапускать по отдельности, можно получить совершено разные результаты

In [21]:
import category_encoders as ce

axis_transformer = Pipeline(steps=[
    ('inputer', SimpleImputer(missing_values=0, strategy='mean')),
    ('polynom', PolynomialFeatures(2,include_bias=False)),
    ('scaler',  StandardScaler())])

clarity_map = [{
    'col':'clarity',##### Обратить внимание
    'mapping':{'FL':10, 'IF':9, 'VVS1':8, 'VVS2':7, 'VS1':6, 'VS2':5, 
               'SI1':4, 'SI2':3, 'I1':2, 'I2':1, 'I3':0}
    }]

clarity_transformer = Pipeline(steps=[
    ('ce',ce.OrdinalEncoder(mapping=clarity_map)),
    ('scaler', MinMaxScaler()),
    ])
    

CT = ColumnTransformer([
        ("pol_std", axis_transformer, axis_features),
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(sparse_output=False), cat_features), ##### Обратить внимание sparse_output=False
        ("ordinal_map", clarity_transformer, ['clarity'])##### Обратить внимание category_encoders
        ]).set_output(transform = 'pandas') ##### Обратить внимание на set_output(transform = 'pandas')

display(CT)

res_ct = CT.fit_transform(X_train)
pd.DataFrame(res_ct).head()


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('pol_std', ...), ('num', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``featu

,pol_std__x,pol_std__y,pol_std__z,pol_std__x^2,pol_std__x y,...,cat__color_G,cat__color_H,cat__color_I,cat__color_J,ordinal_map__clarity
25885,1.056232,0.989051,1.022203,1.023525,0.989719,...,0.0,0.0,0.0,1.0,0.571429
1804,-0.400341,-0.428497,-0.554553,-0.462358,-0.475750,...,0.0,0.0,0.0,0.0,0.428571
1844,-1.159904,-1.186306,-1.191042,-1.079824,-1.092955,...,0.0,0.0,0.0,0.0,0.285714
27396,0.627303,0.650266,0.602699,0.544753,0.560393,...,0.0,0.0,0.0,0.0,0.571429
24691,-1.240328,-1.275460,-1.234439,-1.138886,-1.155230,...,0.0,0.0,0.0,0.0,0.142857


In [20]:
X_train.describe()

,carat,depth,table,x,y,z
count,28318.000000,28318.000000,28318.000000,28318.000000,28318.000000,28318.000000
mean,0.800220,61.751038,57.467547,5.736994,5.739613,3.541982
std,0.473816,1.445905,2.230641,1.121679,1.124265,0.694822
min,0.200000,43.000000,43.000000,0.000000,0.000000,0.000000
25%,0.400000,61.000000,56.000000,4.720000,4.730000,2.920000
50%,0.700000,61.800000,57.000000,5.700000,5.710000,3.530000
75%,1.050000,62.500000,59.000000,6.550000,6.540000,4.040000
max,5.010000,79.000000,95.000000,10.740000,31.800000,6.980000
